In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Loan rate lookups

`rate_changes` records central bank interest rate decisions. `applications` logs loan applications. For each application, attach the rate in effect on that date.

1. Sort both by `date` and merge.
2. Add `monthly_payment = principal * rate / 100 / 12`. Which applicant has the highest monthly payment? Use `np.argmax`.
3. Named-agg groupby by `loan_type`: mean `rate` and total `principal`.
4. Use `np.argsort` to rank loan types by mean rate, lowest to highest.

In [14]:
rate_changes = pd.DataFrame({
    'date': pd.to_datetime(['2022-01-01','2022-05-01','2022-09-01','2023-01-01','2023-06-01']),
    'rate': [0.25, 1.00, 3.00, 4.50, 5.25],
})

applications = pd.DataFrame({
    'date':       pd.to_datetime(['2022-03-15','2022-07-20','2022-11-08','2023-04-01','2023-08-15']),
    'applicant':  ['A', 'B', 'C', 'D', 'E'],
    'principal':  [200000, 150000, 300000, 180000, 250000],
    'loan_type':  ['mortgage', 'personal', 'mortgage', 'personal', 'mortgage'],
})

# Your code here
rate_changes = rate_changes.sort_values('date')
applications  = applications.sort_values('date')
m = pd.merge_asof(applications, rate_changes,
                  on = 'date')

m['monthly_payment'] = m['principal']*m['rate']/100/12

print(m.iloc[np.argmax(m['monthly_payment'])]['applicant'],'has the highest monthly payment')

g = m.groupby('loan_type').agg(
    mean_rate = ('rate','mean'),
    total_principal = ('principal','sum')
)

g.index[np.argsort(g['mean_rate'])]

E has the highest monthly payment


Index(['personal', 'mortgage'], dtype='object', name='loan_type')

---

## Level 2 — Patient vitals at shift handoffs

**New concept: `merge_ordered` with `left_by=`**

Use `left_by=` when the **left** DataFrame has groups and the **right** has common reference points that apply to all groups. It merges each left group with the entire right DataFrame, then combines the results.

```python
pd.merge_ordered(left, right, on='time', left_by='patient_id', fill_method='ffill')
```

This is different from `merge_asof` with `by=` — there, both sides have the same groups and match on them. Here, only the left has groups; the right is a shared reference.

`vitals` records heart rate for two patients at irregular intervals. `shifts` records nurse handoff times — the same schedule for all patients. For each shift change, find the most recent heart rate on file for each patient.

1. Merge with `left_by='patient_id'` and `fill_method='ffill'`.
2. Filter to shift-change rows (where `nurse` is not NaN). This shows the prevailing heart rate at each handoff.
3. Named-agg groupby by `patient_id` on shift rows: mean and max `heart_rate`.
4. Use `np.corrcoef` to check whether P1 and P2's heart rates move together across shift changes.

In [39]:
vitals = pd.DataFrame({
    'time':       pd.to_datetime(['08:00','10:30','14:00','08:00','11:00','15:30'], format='%H:%M'),
    'patient_id': ['P1','P1','P1','P2','P2','P2'],
    'heart_rate': [72, 75, 68, 85, 88, 82],
})

shifts = pd.DataFrame({
    'time':  pd.to_datetime(['09:00','13:00','17:00'], format='%H:%M'),
    'nurse': ['Chen', 'Kim', 'Patel'],
})


# Your code here

m = pd.merge_ordered(
    vitals, 
    shifts, 
    on = 'time',
    left_by = 'patient_id',
    fill_method='ffill'
)
mn = m.dropna(subset=['nurse'])

g = mn.groupby('patient_id').agg(
    mean_hr = ("heart_rate","mean"),
    max_hr = ('heart_rate','max')
)

ns = mn.set_index(['time','patient_id'])['heart_rate'].unstack().dropna()
c = np.corrcoef(ns['P1'],ns['P2'])[0,1]
print(f'correlation is {c:.4f}')
print('highly correlated')

correlation is 0.9966
highly correlated


---

## Level 3 — Two-country GDP

`country_a` and `country_b` report annual GDP growth rates on alternating years. Build a complete timeline and compare the two economies.

1. `merge_ordered` with `fill_method='ffill'`, drop NaN rows.
2. Add `gap = gdp_a - gdp_b`. Which year had the widest gap (check both `.idxmax()` and `.idxmin()`, resolve to actual year with `.loc[]`).
3. `np.corrcoef` on the two GDP columns — do the economies move together?
4. Compute each country's mean growth using `np.mean`. Then collect years where **both** countries were simultaneously above their respective means — use a list comprehension with `zip` over three columns.

In [53]:
country_a = pd.DataFrame({
    'year':  pd.to_datetime(['2015','2017','2019','2021','2023'], format='%Y'),
    'gdp_a': [2.3, 2.8, 1.9, -1.2, 3.1],
})

country_b = pd.DataFrame({
    'year':  pd.to_datetime(['2016','2018','2020','2022'], format='%Y'),
    'gdp_b': [1.8, 2.4, -3.5, 4.2],
})

# Your code here

m = pd.merge_ordered(
    country_a, 
    country_b,
    fill_method='ffill'
).dropna()

m['gap'] = m['gdp_a'] - m['gdp_b']

print(m.loc[m['gap'].idxmax(),'year'],'has the widest gap one direction')

print(m.loc[m['gap'].idxmin(),'year'],'has the widest gap another direction')

c = np.corrcoef(m['gdp_b'],m['gdp_a'])[0,1]
print('correlation is: ', c)
print('slightly correlated')

ma = np.mean(m['gdp_a'])
mb = np.mean(m['gdp_b'])

[y for y,a,b in zip(m['year'],m['gdp_a'],m['gdp_b']) if (a>ma) & (b >mb)]

2020-01-01 00:00:00 has the widest gap one direction
2022-01-01 00:00:00 has the widest gap another direction
correlation is:  0.28746069304527
slightly correlated


[Timestamp('2016-01-01 00:00:00'),
 Timestamp('2017-01-01 00:00:00'),
 Timestamp('2018-01-01 00:00:00'),
 Timestamp('2019-01-01 00:00:00'),
 Timestamp('2023-01-01 00:00:00')]